# Wildlife Intrusion & Poaching Prevention System
### AI-Based Edge Camera Surveillance & Threat Tracking Pipeline

Welcome to the interactive test bench for the **Wildlife Intrusion & Poaching Prevention System**. This system is designed for edge-device deployment in protected forest reserves. It processes video feeds (applying a simulated thermal/infrared night filter), detects targets (animals, humans, vehicles, equipment), tracks their trajectories, evaluates threat severity levels, dispatches mock alerts, and maintains CSV logs for ecological auditing.

---

## 🛠️ Step 1: Environment Setup & Core Imports

We start by loading standard scientific computing and computer vision libraries, alongside the custom edge-surveillance modules:
- `data_manager.py`: Manages storage limits, directory structures, configurations, and CSV ecological logs.
- `alert_system.py`: Emulates SMS transmissions to rangers and compressed satellite communication packages.
- `detector.py`: Wraps YOLOv8, implements centroid trajectory tracking, boundary line crossings, and crops high-contrast evidence snapshots.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

# Import core surveillance modules
from data_manager import DataManager
from alert_system import AlertSystem
from detector import WildlifeDetector
from simulator import generate_synthetic_surveillance_feed

print("✅ All dependencies and custom core modules loaded successfully!")

## ⚙️ Step 2: Initialize Core System Managers

Let's initialize the core system nodes. This setup automatically creates the database structure:
- `data/logs/` for CSV movement logs and transmission text logs.
- `data/snapshots/` for storing compressed evidence images.
- `data/output/` for processing outcomes (videos and charts).

In [ ]:
# Initialize database paths and managers
dm = DataManager()
alert_sys = AlertSystem(dm)
detector = WildlifeDetector(dm, alert_sys)

# Display default configurations
config = dm.get_config()
print("\n--- Camera Node Configurations ---")
print(f"Node ID:             {config['node_id']}")
print(f"GPS Coordinates:     Lat {config['camera_gps']['latitude']}, Lon {config['camera_gps']['longitude']}")
print(f"Perimeter boundary:  {config['perimeter_boundary']['line_type']} at {config['perimeter_boundary']['position_ratio']*100}% height")
print(f"Alert Target SMS:    {config['alert_recipient_sms']}")
print(f"Max Storage Capacity: {config['storage_max_size_mb']} MB")

## 🎬 Step 3: Generate Synthetic Surveillance Feed

To test the system out-of-the-box, we generate a synthetic nighttime surveillance video (`data/synthetic_surveillance.mp4`) simulating a camera feed at a forest boundary line.

The video contains:
1. **An Elephant** (slow-moving animal at the bottom, low threat severity).
2. **A Patrol Vehicle** (fast-moving across the center, medium threat severity).
3. **An Armed Poacher** (human walking top-to-bottom, carrying gear/rifle, crossing the perimeter boundary - critical threat severity).

In [ ]:
# Generate the test video and its frame-by-frame metadata
video_path, metadata_path = generate_synthetic_surveillance_feed(dm.data_dir)

print(f"\n🎥 Video file generated at: {video_path}")
print(f"📝 Ground-truth JSON generated at: {metadata_path}")

## 🚀 Step 4: Run Surveillance Inference Pipeline

Now we process the video. The pipeline will:
1. Convert each frame to a realistic thermal color map (`COLORMAP_INFERNO`).
2. Feed detections into the centroid tracker to compute trajectories.
3. Calculate threat severity levels (LOW, MEDIUM, CRITICAL).
4. Check for perimeter boundary line crossings.
5. Save evidence crops for threats, write CSV logs, and mock alert ranger stations.

In [ ]:
import json
import time

cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Set up Output Video Writer
output_video_path = os.path.join(dm.output_dir, "annotated_synthetic_feed.mp4")
fourcc = cv2.VideoWriter_fourcc(*'avc1')
out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Load the frame metadata to match YOLO predictions with simulated signatures
with open(metadata_path, 'r') as mf:
    frame_metadata = json.load(mf)

# Statistics tracking
inference_times = []
detections_count = {}
timeline_data = []

print("🎥 Analyzing surveillance video... Please wait.")
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Fetch metadata for current frame
    current_metadata = frame_metadata.get(str(frame_idx), None)
    
    # Process frame
    t_start = time.time()
    annotated_frame, tracked_objects = detector.process_frame(
        frame=frame,
        metadata=current_metadata
    )
    inference_times.append(time.time() - t_start)
    
    # Write annotated thermal frame to output video
    out_video.write(annotated_frame)
    
    # Record metrics
    for obj in tracked_objects:
        cls_name = obj['class']
        detections_count[cls_name] = detections_count.get(cls_name, 0) + 1
        
        timeline_data.append({
            'time': frame_idx / fps,
            'class': cls_name,
            'severity': obj['severity']
        })
        
    frame_idx += 1
    if frame_idx % 30 == 0 or frame_idx == total_frames:
        print(f"   Processed {frame_idx}/{total_frames} frames ({(frame_idx/total_frames)*100:.1f}%)")

cap.release()
out_video.release()
print(f"\n🎉 Processing complete! Output saved to: {output_video_path}")
print(f"⏱️ Average Model Latency: {np.mean(inference_times)*1000:.2f} ms ({1.0/np.mean(inference_times):.1f} FPS)")

## 🖼️ Step 5: Visualize High-Contrast Evidence snapshots

For every threat categorized as `MEDIUM` or `CRITICAL` (such as the vehicle intrusion and the armed poacher crossing the perimeter), the detector clips the bounding box, applies a local high-contrast filter (grayscale histogram equalization), and saves it to disk.

Let's display these high-contrast cropped evidence images:

In [ ]:
snapshot_files = [f for f in os.listdir(dm.snapshots_dir) if f.lower().endswith('.jpg')]
print(f"Total evidence snaps saved: {len(snapshot_files)}")

if snapshot_files:
    # Plot snapshots side-by-side
    fig, axes = plt.subplots(1, min(len(snapshot_files), 3), figsize=(15, 5), facecolor='#111827')
    if len(snapshot_files) == 1:
        axes = [axes]
        
    for idx, fn in enumerate(sorted(snapshot_files, reverse=True)[:3]):
        img_path = os.path.join(dm.snapshots_dir, fn)
        img = Image.open(img_path)
        
        axes[idx].imshow(img, cmap='gray')
        axes[idx].set_title(fn.split('_')[1].upper() + " Threat Evidence", color='white', fontsize=12)
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No evidence snapshots available.")

## 📝 Step 6: Ecological Logging and Auditing

A structured CSV log is appended locally for ecological and security reporting. Let's load the logs using Pandas:

In [ ]:
if os.path.exists(dm.log_file):
    df_logs = pd.read_csv(dm.log_file)
    print("\n--- CSV Movement Log Data ---")
    display(df_logs.tail(10))
else:
    print("❌ Ecological CSV log file not found!")

## 📈 Step 7: Plot Threat Detection Timeline

Let's visualize the security alerts triggered during the video. We plot a chronological scatter chart colored by threat level.

In [ ]:
if timeline_data:
    df_timeline = pd.DataFrame(timeline_data)
    
    # Set dark theme styles
    plt.figure(figsize=(12, 5), facecolor='#111827')
    ax = plt.axes()
    ax.set_facecolor('#1F2937')
    
    # Color map
    color_map = {'LOW': '#10B981', 'MEDIUM': '#F59E0B', 'CRITICAL': '#EF4444'} # Green, Amber, Red
    sizes_map = {'LOW': 30, 'MEDIUM': 80, 'CRITICAL': 150}
    
    for sev, group in df_timeline.groupby('severity'):
        plt.scatter(
            group['time'], 
            [sev] * len(group), 
            label=f"{sev} Threat", 
            color=color_map[sev],
            s=sizes_map[sev],
            alpha=0.85,
            edgecolors='white'
        )
        
    plt.title("Surveillance Event Timeline & Threat Distributions", color='white', fontsize=14, pad=15)
    plt.xlabel("Video Timeline (seconds)", color='#9CA3AF', fontsize=11)
    plt.ylabel("Severity Threat Level", color='#9CA3AF', fontsize=11)
    plt.grid(True, color='#374151', linestyle=':', linewidth=0.5)
    ax.tick_params(colors='#6B7280')
    ax.spines['bottom'].set_color('#374151')
    ax.spines['left'].set_color('#374151')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.legend(facecolor='#374151', edgecolor='#4B5563', labelcolor='white')
    
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No timeline data to plot.")

## 🎥 Step 8: Run Detections on Your Custom Input Videos

To test the pipeline on your own custom video files, simply place your video in the workspace, set its filename below, and execute the cell. 

This runs the actual **YOLOv8 deep learning network** to classify humans, vehicles, and wildlife, applying the thermal-conversion filter in real time.

In [ ]:
# REPLACE WITH YOUR OWN VIDEO FILENAME (e.g. "elephant_crossing.mp4")
CUSTOM_INPUT_VIDEO = ""

if CUSTOM_INPUT_VIDEO and os.path.exists(CUSTOM_INPUT_VIDEO):
    print(f"🔍 Custom video found: {CUSTOM_INPUT_VIDEO}. Initializing processing pipeline...")
    
    cap = cv2.VideoCapture(CUSTOM_INPUT_VIDEO)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    f_rate = cap.get(cv2.CAP_PROP_FPS)
    tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    custom_out_name = f"thermal_annotated_{os.path.basename(CUSTOM_INPUT_VIDEO)}"
    custom_out_path = os.path.join(dm.output_dir, custom_out_name)
    
    writer = cv2.VideoWriter(custom_out_path, cv2.VideoWriter_fourcc(*'avc1'), f_rate, (w, h))
    
    f_count = 0
    t_start = time.time()
    
    # Clear tracking history to prevent overlap
    detector.active_tracks.clear()
    detector.logged_tracks.clear()
    
    while True:
        r, f = cap.read()
        if not r:
            break
            
        # Pass None as metadata to run real YOLOv8 object detection on your frame
        ann_f, _ = detector.process_frame(frame=f, metadata=None)
        writer.write(ann_f)
        
        f_count += 1
        if f_count % 30 == 0 or f_count == tot_frames:
            print(f"   Processed {f_count}/{tot_frames} frames ({(f_count/tot_frames)*100:.1f}%%)")
            
    cap.release()
    writer.release()
    
    print(f"\n🎉 Success! Thermal annotated video saved at: {custom_out_path}")
    print(f"🕒 Total processing time: {time.time() - t_start:.2f} seconds")
else:
    print("ℹ️ Please enter a valid path to an existing input video file above to test.")